# LongFlow P1 — E3 eval: 80K polish checkpoint

Runtime: **L4 GPU**. Measures whether the 60K-step continuation (loss 0.965 → 0.693,
`full10k_80k.pt`) moved the numbers that matter. Prior numbers to beat / re-baseline:

| measurement | 20K ckpt (prior) | source |
|---|---|---|
| held-out parity | WER 0.105 / sim 0.824 — n=5, ONE speaker | `eval10k_metrics.json` |
| teacher-forced dispersion (heun8) | 0.941 vs teacher 0.937 (slightly hot) | E1 audit |
| closed-loop drift, short paragraph | 1.03 → 1.21 (8-seg), teacher ≈ 1.16 | E1b |
| endurance drift (1,668-word script) | 1.10 → 1.57 over 5.3 min, hot runaway | endurance test |

This eval also fixes the reviewer's held-out criticism: fresh **multi-speaker**
`test.clean` capture, one utterance per speaker.

LongFlow code comes from GitHub (`Josh-E-S/LongFlow` @ main, commit ≥ b3fcd74 —
needs `latent_stats(segments=8)`); private repo, so authorize GitHub in Colab or
drag `longflow_bundle.zip` in as the fallback. The endurance script is rebuilt
deterministically from the training cache (same recipe as the 2026-07-08 run),
so no script file is needed.

Colab generates; Mac analyzes (WER/ECAPA via `src/eval/metrics.py` on the downloaded zip).


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, sys, time
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"  # held-out, multi-speaker
os.makedirs(EVAL_CACHE_DIR, exist_ok=True)
AUDIO_DIR = "/content/eval80k_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

# LongFlow code: prefer a GitHub clone; fall back to a dragged-in bundle zip.
# Fail loudly on setup problems (2026-07-07 lesson: never let them look like data skips).
if not os.path.exists("/content/LongFlow/src"):
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
if os.path.exists("/content/LongFlow/src"):
    sys.path.insert(0, "/content/LongFlow")
    !cd /content/LongFlow && git log --oneline -1
else:
    assert os.path.exists("/content/longflow_bundle.zip"), (
        "no repo and no bundle: private-repo clone failed (authorize GitHub in Colab "
        "or clone with a token into /content/LongFlow) OR drag longflow_bundle.zip in"
    )
    !cd /content && unzip -q -o longflow_bundle.zip
    sys.path.insert(0, "/content")
from src.cache.capture import SampleCapture, save_utterance
from src.flow_head.cfm import euler_sample, heun_sample
from src.flow_head.integration import FlowHeadPatch
from src.flow_head.trainer import load_checkpoint

CKPT = f"{CKPT_DIR}/full10k_80k.pt"
assert os.path.exists(CKPT), f"missing {CKPT} — check Drive"
head80, mean80, std80 = load_checkpoint(CKPT)
head80 = head80.to("cuda")
print(f"80K head loaded: {sum(p.numel() for p in head80.parameters())/1e6:.1f}M params")

# 20K baseline for A/B — EDIT the name if yours differs; eval degrades gracefully without it
BASELINE_CKPT = f"{CKPT_DIR}/full10k_20k.pt"
if os.path.exists(BASELINE_CKPT):
    head20, mean20, std20 = load_checkpoint(BASELINE_CKPT)
    head20 = head20.to("cuda")
    print("20K baseline loaded")
else:
    head20 = None
    print(f"NOTE: no baseline at {BASELINE_CKPT} — A/B cells will render 80K only. "
          f"Candidates on Drive: {[os.path.basename(f) for f in glob.glob(CKPT_DIR + '/*.pt')]}")

def decode_latents(z):  # z: [T, d_latent] head-space fp32 -> 24 kHz waveform
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi  # verified against generate() source in the gate run
    for shape in (z.unsqueeze(0), z.unsqueeze(0).transpose(1, 2)):
        try:
            out = model.model.acoustic_tokenizer.decode(shape)
            wav = out[0] if isinstance(out, tuple) else out
            return wav.detach().float().cpu().numpy().squeeze()
        except Exception as e:
            print(f"decode attempt {tuple(shape.shape)} failed: {repr(e)[:150]}")
    raise RuntimeError("both decode shapes failed — paste the errors to Claude")

print("READY")


## 1. Held-out multi-speaker capture — resumable, ~20 min

Fresh `test.clean` utterances, **one per speaker, 20 speakers** (the 10K training
cache is `train.clean.360` — zero overlap by construction). Same filters and
`cfg_scale=1.3` as training capture. Saves `.pt` pairs + the 3s voice prompt per
speaker to Drive. Safe to interrupt and re-run.


In [ ]:
import soundfile as sf
from pathlib import Path
from datasets import load_dataset

TARGET_SPEAKERS = 20
ds = load_dataset("mythicinfinity/libritts_r", "clean", split="test.clean", streaming=True)
done_speakers = {f.stem.split("__")[0] for f in Path(EVAL_CACHE_DIR).glob("*.pt")}
print(f"resuming with {len(done_speakers)} speakers already cached")
t0 = time.time()
for ex in ds:
    if len(done_speakers) >= TARGET_SPEAKERS:
        break
    spk = str(ex.get("speaker_id"))
    if spk in done_speakers:
        continue
    text = ex["text_normalized"].strip()
    audio, sr = ex["audio"]["array"], ex["audio"]["sampling_rate"]
    if not (30 <= len(text) <= 180) or len(audio) < 3 * sr:
        continue
    uid = f"{spk}__{str(ex['id']).replace('/', '_')}"
    prompt_path = f"{EVAL_CACHE_DIR}/{uid}_prompt.wav"
    sf.write(prompt_path, audio[: 3 * sr], sr)
    inputs = processor(
        text=[f"Speaker 1: {text}\n"], voice_samples=[[prompt_path]],
        return_tensors="pt", padding=True,
    )
    inputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}
    with SampleCapture(model) as cap, torch.inference_mode():
        model.generate(**inputs, tokenizer=processor.tokenizer, cfg_scale=1.3)
    utt = cap.to_utterance(uid, text, meta={"speaker": spk, "split": "test.clean"})
    save_utterance(utt, f"{EVAL_CACHE_DIR}/{uid}.pt")
    done_speakers.add(spk)
    print(f"{len(done_speakers)}/{TARGET_SPEAKERS}  {uid}  ({(time.time()-t0)/max(1,len(done_speakers)):.0f}s/utt)")
print(f"DONE: {len(done_speakers)} held-out speakers cached")


## 2. Teacher-forced dispersion re-audit (E1 replication on the 80K head)

Marginal std + per-condition spread in standardized space, same protocol as E1.
E1 numbers (20K head): heun8 marginal **0.941** / per-cond 0.631; teacher 0.937.
Question: did 40K more steps change the balance (heun8 ran slightly hot in-loop)?


In [ ]:
eval_files = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*.pt"))
assert eval_files, "run cell 1 first"
hiddens, latents = [], []
for f in eval_files:
    d = torch.load(f, weights_only=True)
    hiddens.append(d["hidden"].float())
    latents.append(d["latent"].float())
H = torch.cat(hiddens).to("cuda")          # [N, d_model]
L = torch.cat(latents).to("cuda")          # [N, d_latent] head-space

heads = {"80k": (head80, mean80, std80)}
if head20 is not None:
    heads["20k"] = (head20, mean20, std20)
N_MARGINAL, N_COND, K = 512, 24, 16
report = {}
for name, (head, mean, std) in heads.items():
    mean_c, std_c = mean.to("cuda"), std.to("cuda")
    Lz = (L - mean_c) / std_c              # teacher latents, standardized w/ this ckpt's stats
    teacher_std = float(Lz[:N_MARGINAL].std())
    report[name] = {"teacher_std": teacher_std, "samplers": {}}
    for tag, fn, nfe in (("heun8", heun_sample, 8), ("euler16", euler_sample, 16), ("euler4", euler_sample, 4)):
        with torch.inference_mode():
            z = fn(head, H[:N_MARGINAL], head.cfg.d_latent, nfe=nfe, sway=0.0)
            marginal = float(z.std())
            cond_rep = H[:N_COND].repeat_interleave(K, dim=0)
            zk = fn(head, cond_rep, head.cfg.d_latent, nfe=nfe, sway=0.0)
            spread = float(zk.view(N_COND, K, -1).std(dim=1).mean())
        report[name]["samplers"][tag] = {"marginal_std": marginal, "per_cond_spread": spread,
                                         "ratio_vs_teacher": marginal / teacher_std}
        print(f"{name} {tag:8s} marginal {marginal:.3f}  per-cond {spread:.3f}  "
              f"ratio {marginal/teacher_std:.3f}  (teacher {teacher_std:.3f})")
with open("/content/eval80k_dispersion.json", "w") as f:
    json.dump(report, f, indent=2)
print("saved eval80k_dispersion.json — E1 reference: 20K heun8 was 0.941/0.631, teacher 0.937")


## 3. Teacher-forced parity renders — multi-speaker A/B audio

6 speakers × {teacher, 80K-heun8, 20K-heun8 if present}. LISTEN for the
"digital cold" / doubled-offset texture — the whole point of the polish run.
Mac computes WER + ECAPA from the zip afterwards; `manifest.json` carries the texts.


In [ ]:
from IPython.display import Audio, display

def render(head, mean, std, hidden, nfe=8):
    with torch.inference_mode():
        z = heun_sample(head, hidden.to("cuda"), head.cfg.d_latent, nfe=nfe, sway=0.0)
    return decode_latents(z * std.to("cuda") + mean.to("cuda"))

manifest = {}
for f in eval_files[:6]:
    d = torch.load(f, weights_only=True)
    tag = d["utt_id"]
    manifest[tag] = d["text"]
    sf.write(f"{AUDIO_DIR}/{tag}_teacher.wav", decode_latents(d["latent"].float()), 24000)
    sf.write(f"{AUDIO_DIR}/{tag}_80k_heun8.wav",
             render(head80, mean80, std80, d["hidden"].float()), 24000)
    if head20 is not None:
        sf.write(f"{AUDIO_DIR}/{tag}_20k_heun8.wav",
                 render(head20, mean20, std20, d["hidden"].float()), 24000)
    print(tag, "|", d["text"][:70])
    for suffix in ("teacher", "80k_heun8") + (("20k_heun8",) if head20 is not None else ()):
        print(" ", suffix)
        display(Audio(f"{AUDIO_DIR}/{tag}_{suffix}.wav"))
with open("/content/eval80k_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)


## 4. Closed loop — short paragraph, drift curve (E1b replication)

Same machinery as E1b: `FlowHeadPatch`, heun8/sway0, fresh noise per frame.
E1b reference (20K head): drift 1.03 → 1.21 (8 segments), teacher overall ≈ 1.16,
head cost 30 ms/frame.


In [ ]:
PARAGRAPH = (
    "The library had been closed for nearly a decade, but the smell of old paper "
    "still filled the entrance hall. Maria ran her fingers along the dusty railing "
    "and remembered the summers she had spent reading in the corner by the tall "
    "windows. Outside, the rain kept a steady rhythm against the glass, patient "
    "and unhurried, as if it too were waiting for the doors to open again. She "
    "climbed the stairs slowly, counting each step the way she had as a child, "
    "and paused at the top to catch her breath before pushing open the reading "
    "room door."
)
PROMPT_WAV = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*_prompt.wav"))[0]
print("voice prompt:", PROMPT_WAV)

def closed_loop(head, mean, std, text, out_wav, nfe=8, prompt=None):
    inputs = processor(
        text=[f"Speaker 1: {text}\n"], voice_samples=[[prompt or PROMPT_WAV]],
        return_tensors="pt", padding=True,
    )
    inputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}
    t0 = time.time()
    with FlowHeadPatch(model, head, mean, std, nfe=nfe, sway=0.0) as patch, torch.inference_mode():
        out = model.generate(**inputs, tokenizer=processor.tokenizer, cfg_scale=1.3)
    wall = time.time() - t0
    try:
        wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    except AttributeError:  # generate() output API drift — decode recorded latents instead
        wav = decode_latents(torch.cat(patch.latents))
    sf.write(out_wav, wav, 24000)
    stats = patch.latent_stats(segments=8)
    stats.update(wall_s=wall, audio_s=len(wav) / 24000,
                 ms_per_frame=1000 * patch.time_s / max(1, patch.calls))
    return stats

stats = closed_loop(head80, mean80, std80, PARAGRAPH, f"{AUDIO_DIR}/closedloop_80k.wav")
print(json.dumps(stats, indent=2))
print("E1b reference (20K): segments 1.03->1.21, teacher overall ~1.16, 30 ms/frame")
display(Audio(f"{AUDIO_DIR}/closedloop_80k.wav"))
with open("/content/eval80k_closedloop.json", "w") as f:
    json.dump(stats, f, indent=2)


## 5. Endurance re-baseline — the number the thermostat probe waits on

Script rebuilt EXACTLY as in the 2026-07-08 run (last 150 cached training utts,
first 100 sentences, ~1,668 words) — word-for-word identical, so the drift
curves compare directly. Previous (20K head, 8 segments):
**[1.099, 1.193, 1.185, 1.249, 1.327, 1.384, 1.542, 1.572]**, 5.3 min audio
(script compressed by speeding up), early stop.
If the slope flattens materially, polish training was a real drift treatment;
if not, the thermostat probe is next with this curve as its baseline.

**Voice comparability:** set `ENDURANCE_VOICE` to the same prompt wav as the
2026-07-08 run if you still have it; default (a test.clean eval prompt) changes
the voice but not the drift mechanics.


In [ ]:
TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
sents = []
for f in sorted(glob.glob(f"{TRAIN_CACHE_DIR}/*.pt"))[-150:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
script = " ".join(sents[:100])
print(f"script: {len(script.split())} words (2026-07-08 run: 1668)")

ENDURANCE_VOICE = PROMPT_WAV  # <- point at the 2026-07-08 run's voice prompt for a strict A/B
endur = closed_loop(head80, mean80, std80, script, f"{AUDIO_DIR}/endurance_80k.wav",
                    prompt=ENDURANCE_VOICE)
print(json.dumps(endur, indent=2))
print("previous (20K): [1.099, 1.193, 1.185, 1.249, 1.327, 1.384, 1.542, 1.572], 5.3 min audio")
print("(healthy ~ steady 1.0-1.2; climbing past ~1.4 = compounding; sinking = the old fade)")
display(Audio(f"{AUDIO_DIR}/endurance_80k.wav"))
with open("/content/eval80k_endurance.json", "w") as f:
    json.dump(endur, f, indent=2)


## 6. Bundle for the Mac

Download `eval80k_bundle.zip`; on the Mac, WER/ECAPA via `src/eval/metrics.py`
(Whisper reference = each `*_teacher.wav` transcript, per `manifest.json` texts).


In [ ]:
import zipfile
with zipfile.ZipFile("/content/eval80k_bundle.zip", "w") as z:
    for f in glob.glob(f"{AUDIO_DIR}/*.wav"):
        z.write(f, f"audio/{os.path.basename(f)}")
    for f in glob.glob("/content/eval80k_*.json"):
        z.write(f, os.path.basename(f))
print("DONE -> download /content/eval80k_bundle.zip and drop it on the Mac session")
